# 手撕 Attention Sink (StreamingLLM)

## 背景
StreamingLLM 发现：保留前 k 个 token 的 KV Cache（sink tokens）+ 滑动窗口，
即可在长序列上稳定推理。sink tokens 充当"注意力吸收槽"。

## 考察点
- sink tokens 的作用（吸收多余注意力权重）
- 滑动窗口 + sink 的 cache 管理
- 与朴素长序列推理的对比

In [ ]:
import torch
import torch.nn.functional as F
import math

class SinkWindowAttention:
    def __init__(self, n_sink=4, window_size=32, d_head=16):
        self.n_sink = n_sink
        self.window = window_size
        self.d_head = d_head
        self.k_cache = []
        self.v_cache = []

    def _get_cache(self):
        # 保留 sink + 最近 window 个
        if len(self.k_cache) > self.n_sink + self.window:
            self.k_cache = self.k_cache[:self.n_sink] + self.k_cache[-self.window:]
            self.v_cache = self.v_cache[:self.n_sink] + self.v_cache[-self.window:]
        k = torch.stack(self.k_cache, dim=0)  # (cache_len, d_head)
        v = torch.stack(self.v_cache, dim=0)
        return k, v

    def step(self, q, k, v):
        # q,k,v: (d_head,) 单个 token
        self.k_cache.append(k)
        self.v_cache.append(v)
        k_all, v_all = self._get_cache()
        scores = q @ k_all.T / math.sqrt(self.d_head)
        attn = F.softmax(scores, dim=-1)
        return attn @ v_all

In [ ]:
# 验证长序列推理
torch.manual_seed(42)
d_head = 16
sink_attn = SinkWindowAttention(n_sink=4, window_size=16, d_head=d_head)
# 模拟 200 步推理，cache 应被限制
outputs = []
for t in range(200):
    q = torch.randn(d_head)
    k = torch.randn(d_head)
    v = torch.randn(d_head)
    out = sink_attn.step(q, k, v)
    outputs.append(out)
# cache 应被限制在 n_sink + window = 20
assert len(sink_attn.k_cache) == 4 + 16, f"cache 应=20, 实际={len(sink_attn.k_cache)}"
print(f"推理 200 步后 cache 大小: {len(sink_attn.k_cache)} (应=20)")
print(f"输出范数稳定: {outputs[-1].norm().item():.4f}")
print("✅ Attention Sink + 滑动窗口验证通过")